In [6]:
import pandas as pd

# List of penalty files
files = [
    'penalties2015.csv',
    'penalties2016.csv',
    'penalties2017.csv',
    'penalties2018.csv',
    'penalties2019.csv'
]

all_years = []  # to collect results

for file in files:
    # Load
    df = pd.read_csv(file)

    # Clean provnum and Year
    df['provnum'] = pd.to_numeric(df['provnum'], errors='coerce')
    df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
    df = df.dropna(subset=['provnum', 'Year'])
    df['provnum'] = df['provnum'].astype(int)
    df['Year'] = df['Year'].astype(int)

    # Clean fine_amt
    df['fine_amt'] = pd.to_numeric(df['fine_amt'], errors='coerce').fillna(0)

    # Count penalties (pivot)
    penalty_counts = pd.pivot_table(
        df,
        index=['provnum', 'Year'],
        columns='pnlty_type',
        aggfunc='size',
        fill_value=0
    ).reset_index()

    # Sum fines
    fine_sums = df.groupby(['provnum', 'Year'])['fine_amt'].sum().reset_index()

    # Merge counts and fine sums
    merged = pd.merge(penalty_counts, fine_sums, on=['provnum', 'Year'], how='left')

    # Clean column names
    merged.columns = [col.strip().replace(' ', '_') if isinstance(col, str) else col for col in merged.columns]
    merged = merged.rename(columns={
        'Payment_Denial': 'Payment_Denial', 
        'Fine': 'Fine', 
        'fine_amt': 'Total_Fine_Amount'
    })

    # Append to list
    all_years.append(merged)

# Combine all years together
final_penalty_data = pd.concat(all_years, ignore_index=True)

# Fill missing values (if any)
final_penalty_data[['Fine', 'Payment_Denial']] = final_penalty_data[['Fine', 'Payment_Denial']].fillna(0).astype(int)
final_penalty_data['Total_Fine_Amount'] = final_penalty_data['Total_Fine_Amount'].fillna(0)

# Show sample
print(final_penalty_data.head())

# Save to CSV if you want
final_penalty_data.to_csv('penalty_summary_2015_2019.csv', index=False)


   provnum  Year  Fine  Payment_Denial  Total_Fine_Amount
0    15019  2015     1               0             6692.0
1    15037  2015     1               0            13813.0
2    15053  2015     1               1           142870.0
3    15060  2015     2               0            64611.0
4    15076  2015     1               0             1755.0


In [ ]:


# Load your previous merged file
prev_merged = pd.read_csv('merged_PrDeQU_data.csv')  # <-- change filename if needed

# Load the penalty summary (the one we just created)
penalty_summary = pd.read_csv('penalty_summary_2015_2019.csv')

print(penalty_summary.dtypes)

# Make sure keys are numeric
prev_merged['provnum'] = pd.to_numeric(prev_merged['provnum'], errors='coerce').astype('Int64')
prev_merged['Year'] = pd.to_numeric(prev_merged['Year'], errors='coerce').astype('Int64')

penalty_summary['provnum'] = pd.to_numeric(penalty_summary['provnum'], errors='coerce').astype('Int64')
penalty_summary['Year'] = pd.to_numeric(penalty_summary['Year'], errors='coerce').astype('Int64')

# Merge
final_merged = pd.merge(
    prev_merged,
    penalty_summary,
    on=['provnum', 'Year'],
    how='left'
)

# If no penalties, fill counts and fines with 0
final_merged['Fine'] = final_merged['Fine'].fillna(0).astype(int)
final_merged['Payment_Denial'] = final_merged['Payment_Denial'].fillna(0).astype(int)
final_merged['Total_Fine_Amount'] = final_merged['Total_Fine_Amount'].fillna(0)

# Save or view
print(final_merged.head())
final_merged.to_csv('merged_PrDeQuPe_data.csv', index=False)


provnum                int64
Year                   int64
Fine                   int64
Payment_Denial         int64
Total_Fine_Amount    float64
dtype: object
